In [0]:
customers = spark.read.csv(
    "/Volumes/workspace/default/retail_data/customers.csv",
    header=True,
    inferSchema=True
)

orders = spark.read.csv(
    "/Volumes/workspace/default/retail_data/orders.csv",
    header=True,
    inferSchema=True
)

products = spark.read.csv(
    "/Volumes/workspace/default/retail_data/products.csv",
    header=True,
    inferSchema=True
)

In [0]:
display(customers)
display(orders)
display(products)

customer_id,customer_name,city
1,Alice,Mumbai
2,Bob,Delhi
3,Charlie,Bengaluru
4,Diana,Pune
5,Ethan,Hyderabad


order_id,customer_id,product_id,quantity
1001,1,101,1
1002,2,102,2
1003,1,103,1
1004,3,104,1
1005,4,105,2
1006,5,101,1
1007,2,105,1
1008,3,102,3


product_id,product_name,price
101,Laptop,80000
102,Mouse,800
103,Keyboard,1500
104,Monitor,12000
105,Headphones,2500


In [0]:
customers.printSchema()
orders.printSchema()
products.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: integer (nullable = true)



In [0]:
customers.write.mode("overwrite").format("delta").saveAsTable("bronze_customers")

orders.write.mode("overwrite").format("delta").saveAsTable("bronze_orders")

products.write.mode("overwrite").format("delta").saveAsTable("bronze_products")

In [0]:
%sql 
SHOW TABLES;

database,tableName,isTemporary
default,bronze_customers,false
default,bronze_orders,false
default,bronze_products,false
default,gold_sales,false
default,silver_orders,false


In [0]:
bronze_customers = spark.table("bronze_customers")
bronze_orders = spark.table("bronze_orders")
bronze_products = spark.table("bronze_products")

In [0]:
silver_orders = (
    bronze_orders
    .dropDuplicates()
    .na.fill({"quantity": 1})
)

In [0]:
silver_orders.write.mode("overwrite").format("delta").saveAsTable("silver_orders")

In [0]:
%sql
SELECT * FROM silver_orders;

order_id,customer_id,product_id,quantity
1002,2,102,2
1004,3,104,1
1003,1,103,1
1005,4,105,2
1007,2,105,1
1001,1,101,1
1006,5,101,1
1008,3,102,3


In [0]:
from pyspark.sql.functions import col

gold_sales = (
    silver_orders
    .join(bronze_customers, "customer_id")
    .join(bronze_products, "product_id")
    .withColumn("revenue", col("price") * col("quantity"))
)

In [0]:
display(gold_sales)

product_id,customer_id,order_id,quantity,customer_name,city,product_name,price,revenue
102,2,1002,2,Bob,Delhi,Mouse,800,1600
104,3,1004,1,Charlie,Bengaluru,Monitor,12000,12000
103,1,1003,1,Alice,Mumbai,Keyboard,1500,1500
105,4,1005,2,Diana,Pune,Headphones,2500,5000
105,2,1007,1,Bob,Delhi,Headphones,2500,2500
101,1,1001,1,Alice,Mumbai,Laptop,80000,80000
101,5,1006,1,Ethan,Hyderabad,Laptop,80000,80000
102,3,1008,3,Charlie,Bengaluru,Mouse,800,2400


In [0]:
gold_sales.write.mode("overwrite").format("delta").saveAsTable("gold_sales")

In [0]:
%sql
SELECT
    product_name,
    SUM(revenue) AS total_revenue
FROM gold_sales
GROUP BY product_name
ORDER BY total_revenue DESC;

product_name,total_revenue
Laptop,160000
Monitor,12000
Headphones,7500
Mouse,4000
Keyboard,1500


In [0]:
%sql
SELECT
    city,
    SUM(revenue) AS revenue
FROM gold_sales
GROUP BY city
ORDER BY revenue DESC;

city,revenue
Mumbai,81500
Hyderabad,80000
Bengaluru,14400
Pune,5000
Delhi,4100


In [0]:
%sql
SELECT
    customer_name,
    SUM(revenue) AS spent
FROM gold_sales
GROUP BY customer_name
ORDER BY spent DESC;

customer_name,spent
Alice,81500
Ethan,80000
Charlie,14400
Diana,5000
Bob,4100


In [0]:
%sql
DESCRIBE HISTORY gold_sales;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
9,2026-07-24T12:43:16.000Z,76002496258078,pandeyshrestha2111@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""6df89651-9d0f-4122-8feb-3d946cd43b6e""}, statsOnLoad -> true)",null,List(1812703075464072),8b9692c4-d93e-4f94-bf8a-1b8a361ba9aa,0724-120239-uxafu3d4-v2n,8,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2919, numDeletionVectorsRemoved -> 0, numOutputRows -> 8, numOutputBytes -> 2937)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
8,2026-07-23T19:27:59.000Z,76002496258078,pandeyshrestha2111@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)","List(1114025757790656, Retail Sales Pipeline, 1008381591368138, 71196024210918, 76002496258078, manual)",List(1812703075464072),68c86d7e-9fcf-43c2-9481-d6c86d2b48cd,0723-191641-4czk7w4c-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2989, p25FileSize -> 2919, numDeletionVectorsRemoved -> 1, minFileSize -> 2919, numAddedFiles -> 1, maxFileSize -> 2919, p75FileSize -> 2919, p50FileSize -> 2919, numAddedBytes -> 2919)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
7,2026-07-23T19:27:58.000Z,76002496258078,pandeyshrestha2111@gmail.com,DELETE,"Map(predicate -> [""(product_name#16024 = Laptop)""])","List(1114025757790656, Retail Sales Pipeline, 1008381591368138, 71196024210918, 76002496258078, manual)",List(1812703075464072),68c86d7e-9fcf-43c2-9481-d6c86d2b48cd,0723-191641-4czk7w4c-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1365, numDeletionVectorsUpdated -> 0, numDeletedRows -> 2, scanTimeMs -> 957, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 407)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
6,2026-07-23T19:27:49.000Z,76002496258078,pandeyshrestha2111@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""6df89651-9d0f-4122-8feb-3d946cd43b6e""}, statsOnLoad -> true)","List(1114025757790656, Retail Sales Pipeline, 1008381591368138, 71196024210918, 76002496258078, manual)",List(1812703075464072),7324e4ae-5357-41b1-a590-c98f532f61ba,0723-191641-4czk7w4c-v2n,5,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2921, numDeletionVectorsRemoved -> 0, numOutputRows -> 8, numOutputBytes -> 2989)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-07-23T19:27:07.000Z,76002496258078,pandeyshrestha2111@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)","List(1114025757790656, Retail Sales Pipeline, 615659967115567, 1019710197506671, 76002496258078, manual)",List(1812703075464072),dc5ae556-c197-4ebe-97e2-02709139c28c,0723-191641-4czk7w4c-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2991, p25FileSize -> 2921, numDeletionVectorsRemoved -> 1, minFileSize -> 2921, numAddedFiles -> 1, maxFileSize -> 2921, p75FileSize -> 2921, p50FileSize -> 2921, numAddedBytes -> 2921)",null,Databricks-Runtime/18.x-aarch64-photon

In [0]:
%sql
DELETE FROM gold_sales
WHERE product_name='Laptop';

num_affected_rows
2


In [0]:
%sql
SELECT * FROM gold_sales;

product_id,customer_id,order_id,quantity,customer_name,city,product_name,price,revenue
102,2,1002,2,Bob,Delhi,Mouse,800,1600
104,3,1004,1,Charlie,Bengaluru,Monitor,12000,12000
103,1,1003,1,Alice,Mumbai,Keyboard,1500,1500
105,4,1005,2,Diana,Pune,Headphones,2500,5000
105,2,1007,1,Bob,Delhi,Headphones,2500,2500
102,3,1008,3,Charlie,Bengaluru,Mouse,800,2400
